# DATA CLEANING AND NORMALIZATION

## confs

In [1]:
from pyspark.sql import SparkSession
from pathlib import Path
import importlib
import data_mgmt_lib_common
import os

importlib.reload(data_mgmt_lib_common)
create_unified_df = data_mgmt_lib_common.create_unified_df

In [2]:
PARENT_FOLDER = Path(os.path.abspath("")).parent.parent

COLS_TO_DROP = ['VendorID', 'tpep_dropoff_datetime', 'store_and_fwd_flag', 'payment_type', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
                'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee']

In [3]:
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 4g pyspark-shell"
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/06 02:49:26 WARN Utils: Your hostname, LAZ, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/06 02:49:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/06 02:49:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = create_unified_df(mode="spark", spark=spark)

reading yellow_tripdata_taxi_data_2026_1.parquet
reading yellow_tripdata_taxi_data_2026_2.parquet
reading yellow_tripdata_taxi_data_2026_3.parquet
reading yellow_tripdata_taxi_data_2026_4.parquet
reading yellow_tripdata_taxi_data_2026_5.parquet


## cleaning

In [5]:
# rate code na is actually 99
df = df.na.fill({'RatecodeID': 99})

df.select('RatecodeID').distinct().show()

+----------+
|RatecodeID|
+----------+
|         6|
|         5|
|         1|
|         3|
|         2|
|         4|
|        99|
+----------+



## removing unneccessary columns

In [ ]:
df = df.drop(*COLS_TO_DROP)

## saving cleaned data to parquet

In [6]:
print(f"df size: {df.count()}")
df.write.parquet(path=str(PARENT_FOLDER / "taxi_data" / "yellow_tripdata_cleaned" / "yellow_tripdata_cleaned.parquet"), mode='overwrite')

df size: 18999282
